In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root so we can import src
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Data balancing
from imblearn.over_sampling import SMOTE

# Scaling
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

import matplotlib.pyplot as plt

In [2]:
RAW_DIR = "../data/raw"

df_feats, feature_cols = get_features(RAW_DIR)

print("Dataset shape:", df_feats.shape)
df_feats.head()

Dataset shape: (76833, 79)


,player_id,minutes_played,goals,assists,yellow_cards,second_yellow_cards,direct_red_cards,penalty_goals,matches_played,clean_sheets,...,won_champions,team_ucl_strength,Titles,win_rate,goals_per_game,num_trophies,ballon_dor_winner,player_name,position,main_position
0,1,153.0,1.0,2,0,0,0,0,6,0,...,0.0,0.005587,0.0,0.5,1.5,0.0,0,Silvio Adzic (1),Attack - Right Winger,Attack
1,1,0.0,0.0,0,0,0,0,0,2,0,...,0.0,0.005587,0.0,0.5,1.5,0.0,0,Silvio Adzic (1),Attack - Right Winger,Attack
2,1,0.0,0.0,0,0,0,0,0,4,0,...,0.0,0.005587,0.0,0.5,1.5,0.0,0,Silvio Adzic (1),Attack - Right Winger,Attack
3,10,0.0,0.0,0,0,0,0,0,2,0,...,0.0,0.005587,0.0,0.5,1.5,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack
4,10,245.0,9.0,3,4,0,1,0,29,0,...,0.0,0.005587,0.0,0.5,1.5,0.0,0,Miroslav Klose (10),Attack - Centre-Forward,Attack


In [3]:
MIN_MINUTES = 100

df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= MIN_MINUTES)
].copy()

# Temporal split: HARD STRICT NON-LEAKING
df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) & (df_ml["season_end_year"] <= 2022)].copy()
df_test  = df_ml[df_ml["season_end_year"] >= 2023].copy()

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

# Split matrices
X_train = df_train[feature_cols]
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[feature_cols]
y_val = df_val["ballon_dor_winner"].astype(int)

X_test = df_test[feature_cols]
y_test = df_test["ballon_dor_winner"].astype(int)


Train: (14301, 79)
Val: (6038, 79)
Test: (3103, 79)


In [4]:
print("Before SMOTE:", y_train.value_counts())

sm = SMOTE(k_neighbors=1, random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_res.value_counts())

# SCALE everything
scaler = StandardScaler()
X_train_res_scaled = scaler.fit_transform(X_train_res)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Before SMOTE: ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64
After SMOTE: ballon_dor_winner
0    14290
1    14290
Name: count, dtype: int64


In [ ]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_res_scaled, y_train_res)

pred_lr = lr.predict(X_val_scaled)
proba_lr = lr.predict_proba(X_val_scaled)[:,1]

print("LOGISTIC VALIDATION METRICS")
print("Recall:", recall_score(y_val, pred_lr))
print("Precision:", precision_score(y_val, pred_lr))
print("F1:", f1_score(y_val, pred_lr))
print("AUC:", roc_auc_score(y_val, proba_lr))


=== LOGISTIC VALIDATION ===
Recall: 0.6666666666666666
Precision: 0.18181818181818182
F1: 0.2857142857142857
AUC: 0.9992819663076499


In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    random_state=42
)

rf.fit(X_train_res_scaled, y_train_res)

pred_rf = rf.predict(X_val_scaled)
proba_rf = rf.predict_proba(X_val_scaled)[:,1]

print("RANDOM FOREST VALIDATION METRICS")
print("Recall:", recall_score(y_val, pred_rf))
print("Precision:", precision_score(y_val, pred_rf))
print("F1:", f1_score(y_val, pred_rf))
print("AUC:", roc_auc_score(y_val, proba_rf))


=== RANDOM FOREST VALIDATION ===
Recall: 0.3333333333333333
Precision: 1.0
F1: 0.5
AUC: 0.99961336647335


In [7]:
scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_res_scaled, y_train_res)

pred_xgb = xgb.predict(X_val_scaled)
proba_xgb = xgb.predict_proba(X_val_scaled)[:,1]

print("XGBOOST VALIDATION METRICS")
print("Recall:", recall_score(y_val, pred_xgb))
print("Precision:", precision_score(y_val, pred_xgb))
print("F1:", f1_score(y_val, pred_xgb))
print("AUC:", roc_auc_score(y_val, proba_xgb))


XGBOOST VALIDATION METRICS
Recall: 0.3333333333333333
Precision: 0.3333333333333333
F1: 0.3333333333333333
AUC: 0.9989505661419497


In [8]:

model_scores = {
    "lr": roc_auc_score(y_val, proba_lr),
    "rf": roc_auc_score(y_val, proba_rf),
    "xgb": roc_auc_score(y_val, proba_xgb),
}

best_model_name = max(model_scores, key=model_scores.get)
best_model = {"lr": lr, "rf": rf, "xgb": xgb}[best_model_name]
best_proba_val = {"lr": proba_lr, "rf": proba_rf, "xgb": proba_xgb}[best_model_name]

print("BEST MODEL:", best_model_name.upper(), "with AUC:", model_scores[best_model_name])


BEST MODEL: RF with AUC: 0.99961336647335


In [10]:
pred_test = best_model.predict(X_test_scaled)
proba_test = best_model.predict_proba(X_test_scaled)[:,1]

print("TEST RESULTS FOR BEST MODEL:", best_model_name.upper())
print("Recall:", recall_score(y_test, pred_test))
print("Precision:", precision_score(y_test, pred_test))
print("F1:", f1_score(y_test, pred_test))
print("AUC:", roc_auc_score(y_test, proba_test))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_test))



TEST RESULTS FOR BEST MODEL: RF
Recall: 0.0
Precision: 0.0
F1: 0.0
AUC: 0.9277652370203161

Confusion Matrix:
[[3101    0]
 [   2    0]]


c:\Users\leodo\OneDrive\Escritorio\machine learning\Machine-learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:

df_test["proba_win"] = proba_test

top_candidates = (
    df_test
    [["player_id","player_name", "season_end_year", "proba_win"]]
    .sort_values("proba_win", ascending=False)
    .head(20)
)

top_candidates


,player_id,player_name,season_end_year,proba_win
6157,132098,Harry Kane (132098),2024.0,0.118038
10788,161056,Joshua Kimmich (161056),2024.0,0.079000
44315,418560,Erling Haaland (418560),2023.0,0.070667
12311,169880,Giovanni Di Lorenzo (169880),2024.0,0.070667
1307,106987,Marcel Sabitzer (106987),2024.0,0.058333
34715,342229,Kylian Mbappé (342229),2024.0,0.055967
34714,342229,Kylian Mbappé (342229),2023.0,0.048167
38592,369081,Federico Valverde (369081),2024.0,0.043433
4536,126414,Hakan Çalhanoğlu (126414),2023.0,0.042000
31515,321528,Denzel Dumfries (321528),2023.0,0.039333
